In [0]:
from pyspark.sql import functions as F

HIERARCHY_TABLE = "workspace.firmable_v2_silver.employee_hierarchy"
EMPLOYEE_TABLE = "workspace.firmable_v2_silver.employees_clean"
ASSIGNMENT_TABLE = "workspace.firmable_v2_silver.manager_assignments"

VALIDATION_TABLE = "workspace.firmable_v2_silver.hierarchy_validation_summary"

hierarchy = spark.table(HIERARCHY_TABLE)
employees = spark.table(EMPLOYEE_TABLE)
assignments = spark.table(ASSIGNMENT_TABLE)

# --------------------------------------------------
# 1. BASIC COUNTS AND UNIQUENESS
# --------------------------------------------------

total_employees = hierarchy.count()

duplicate_employees = (
    hierarchy.groupBy("company_id", "employee_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

missing_employee_records = (
    employees.select("company_id", "employee_id")
    .dropDuplicates()
    .join(
        hierarchy.select("company_id", "employee_id"),
        ["company_id", "employee_id"],
        "left_anti"
    )
    .count()
)

extra_employee_records = (
    hierarchy.select("company_id", "employee_id")
    .dropDuplicates()
    .join(
        employees.select("company_id", "employee_id").dropDuplicates(),
        ["company_id", "employee_id"],
        "left_anti"
    )
    .count()
)

# --------------------------------------------------
# 2. MANAGER REFERENCE VALIDATION
# --------------------------------------------------

manager_lookup = hierarchy.select(
    "company_id",
    F.col("employee_id").alias("checked_manager_id"),
    F.col("root_employee_id").alias("manager_root_id"),
    F.col("hierarchy_depth").alias("manager_depth"),
    F.col("inferred_seniority_level").alias("manager_level")
)

children = hierarchy.filter(
    F.col("manager_id").isNotNull()
)

manager_check = children.alias("child").join(
    manager_lookup.alias("manager"),
    (
        F.col("child.company_id") == F.col("manager.company_id")
    ) & (
        F.col("child.manager_id") == F.col("manager.checked_manager_id")
    ),
    "left"
)

invalid_manager_references = manager_check.filter(
    F.col("manager.checked_manager_id").isNull()
).count()

# --------------------------------------------------
# 3. SENIORITY VALIDATION
# Lower seniority number means more senior.
# --------------------------------------------------

invalid_seniority = manager_check.filter(
    F.col("manager.checked_manager_id").isNotNull() &
    (
        F.col("manager.manager_level") >=
        F.col("child.inferred_seniority_level")
    )
).count()

# --------------------------------------------------
# 4. HIERARCHY DEPTH VALIDATION
# Every child must be exactly one level below its manager.
# --------------------------------------------------

invalid_depth = manager_check.filter(
    F.col("manager.checked_manager_id").isNotNull() &
    (
        F.col("child.hierarchy_depth") !=
        F.col("manager.manager_depth") + 1
    )
).count()

# --------------------------------------------------
# 5. ROOT CONSISTENCY
# Every component root must point to itself.
# Every non-root employee must have a valid root.
# --------------------------------------------------

invalid_root_records = hierarchy.filter(
    (F.col("is_root") & (
        (F.col("root_employee_id") != F.col("employee_id")) |
        (F.col("hierarchy_depth") != 0)
    )) |
    ((~F.col("is_root")) & (
        F.col("root_employee_id").isNull() |
        (F.col("hierarchy_depth") <= 0)
    ))
).count()

root_lookup = hierarchy.filter(
    F.col("is_root")
).select(
    "company_id",
    F.col("employee_id").alias("checked_root_id")
)

invalid_root_references = (
    hierarchy
    .join(
        root_lookup,
        (
            hierarchy.company_id == root_lookup.company_id
        ) & (
            hierarchy.root_employee_id == root_lookup.checked_root_id
        ),
        "left"
    )
    .filter(F.col("checked_root_id").isNull())
    .count()
)

# Every employee must share the same root as their direct manager.
invalid_component_links = manager_check.filter(
    F.col("manager.checked_manager_id").isNotNull() &
    (
        F.col("child.root_employee_id") !=
        F.col("manager.manager_root_id")
    )
).count()

# --------------------------------------------------
# 6. SELF-RELATIONSHIP AND CYCLE SAFETY
# --------------------------------------------------

self_relationships = hierarchy.filter(
    F.col("employee_id") == F.col("manager_id")
).count()

# Strictly increasing seniority along every manager-to-child
# edge makes a directed cycle impossible.
# The seniority validation above checks this condition.

# --------------------------------------------------
# 7. ASSIGNMENT RECONCILIATION
# --------------------------------------------------

assigned_in_hierarchy = hierarchy.filter(
    F.col("manager_id").isNotNull()
).count()

assigned_in_source = assignments.filter(
    F.col("candidate_manager_id").isNotNull()
).count()

assignment_count_difference = abs(
    assigned_in_hierarchy - assigned_in_source
)

# --------------------------------------------------
# 8. ROOT AND DEPTH STATISTICS
# --------------------------------------------------

root_count = hierarchy.filter(F.col("is_root")).count()

ceo_roots = hierarchy.filter(
    F.col("root_type") == "CEO_ROOT"
).count()

unassigned_roots = hierarchy.filter(
    F.col("root_type") == "UNASSIGNED_ROOT"
).count()

maximum_depth = hierarchy.agg(
    F.max("hierarchy_depth")
).first()[0]

# --------------------------------------------------
# 9. BUILD VALIDATION REPORT
# --------------------------------------------------

checks = [
    ("duplicate_employee_records", duplicate_employees),
    ("missing_employee_records", missing_employee_records),
    ("extra_employee_records", extra_employee_records),
    ("invalid_manager_references", invalid_manager_references),
    ("invalid_seniority_relationships", invalid_seniority),
    ("invalid_hierarchy_depth", invalid_depth),
    ("invalid_root_records", invalid_root_records),
    ("invalid_root_references", invalid_root_references),
    ("invalid_component_links", invalid_component_links),
    ("self_relationships", self_relationships),
    ("assignment_count_difference", assignment_count_difference)
]

validation_rows = [
    (check_name, int(check_value), check_value == 0)
    for check_name, check_value in checks
]

validation_report = spark.createDataFrame(
    validation_rows,
    ["validation_check", "failure_count", "passed"]
)

(
    validation_report.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(VALIDATION_TABLE)
)

# --------------------------------------------------
# 10. FINAL OUTPUT
# --------------------------------------------------

print("=" * 60)
print("HIERARCHY VALIDATION REPORT")
print("=" * 60)

validation_report.orderBy(
    "validation_check"
).show(truncate=False)

print("\nHIERARCHY SUMMARY")
print("Total employees:", total_employees)
print("Assigned employees:", assigned_in_hierarchy)
print("Component roots:", root_count)
print("CEO roots:", ceo_roots)
print("Unassigned roots:", unassigned_roots)
print("Maximum hierarchy depth:", maximum_depth)

print("\nROOT DISTRIBUTION")
hierarchy.filter(
    F.col("is_root")
).groupBy(
    "root_type"
).count().show(truncate=False)

print("\nDEPTH DISTRIBUTION")
hierarchy.groupBy(
    "hierarchy_depth"
).count().orderBy(
    "hierarchy_depth"
).show()

failed_checks = sum(
    1 for _, value in checks if value != 0
)

assert failed_checks == 0, (
    f"Hierarchy validation failed: {failed_checks} checks have errors."
)

print("\nOUTPUT TABLE:", VALIDATION_TABLE)
print("ALL HIERARCHY VALIDATION CHECKS PASSED")

In [0]:
from pyspark.sql import functions as F

GROUND_TRUTH_TABLE = "workspace.firmable_v2_bronze.ground_truth"
HIERARCHY_TABLE = "workspace.firmable_v2_silver.employee_hierarchy"
COMPANY_TABLE = "workspace.firmable_v2_bronze.companies"

METRICS_TABLE = "workspace.firmable_v2_evaluation.manager_assignment_metrics"
COMPANY_METRICS_TABLE = "workspace.firmable_v2_evaluation.company_structure_metrics"

# --------------------------------------------------
# 1. LOAD EVALUATION DATA
# Ground truth is accessed only in this notebook.
# --------------------------------------------------

ground_truth = spark.table(GROUND_TRUTH_TABLE)
hierarchy = spark.table(HIERARCHY_TABLE)
companies = spark.table(COMPANY_TABLE)

gt_columns = {c.lower(): c for c in ground_truth.columns}

employee_candidates = [
    "employee_id",
    "employee_identifier"
]

manager_candidates = [
    "actual_manager_id",
    "manager_id",
    "true_manager_id",
    "ground_truth_manager_id",
    "manager_employee_id",
    "reports_to_employee_id",
    "parent_employee_id"
]

employee_col = next(
    (gt_columns[c] for c in employee_candidates if c in gt_columns),
    None
)

manager_col = next(
    (gt_columns[c] for c in manager_candidates if c in gt_columns),
    None
)

company_col = gt_columns.get("company_id")

assert employee_col is not None, (
    f"Could not identify employee ID in ground truth. "
    f"Available columns: {ground_truth.columns}"
)

assert manager_col is not None, (
    f"Could not identify manager ID in ground truth. "
    f"Available columns: {ground_truth.columns}"
)

assert company_col is not None, (
    f"Ground truth must contain company_id. "
    f"Available columns: {ground_truth.columns}"
)

print("GROUND TRUTH COLUMN MAPPING")
print("Company ID:", company_col)
print("Employee ID:", employee_col)
print("True manager ID:", manager_col)

# --------------------------------------------------
# 2. STANDARDIZE GROUND TRUTH AND PREDICTIONS
# --------------------------------------------------

truth = ground_truth.select(
    F.col(company_col).alias("company_id"),
    F.col(employee_col).alias("employee_id"),
    F.col(manager_col).alias("true_manager_id")
)

predictions = hierarchy.select(
    "company_id",
    "employee_id",
    F.col("manager_id").alias("predicted_manager_id"),
    "assignment_status",
    "candidate_score",
    "root_type"
)

# --------------------------------------------------
# 3. DATA INTEGRITY CHECKS
# --------------------------------------------------

truth_duplicates = (
    truth.groupBy("company_id", "employee_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

prediction_duplicates = (
    predictions.groupBy("company_id", "employee_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert truth_duplicates == 0, (
    f"Ground truth contains {truth_duplicates} duplicate employees."
)

assert prediction_duplicates == 0, (
    f"Predictions contain {prediction_duplicates} duplicate employees."
)

missing_predictions = (
    truth.select("company_id", "employee_id")
    .join(
        predictions.select("company_id", "employee_id"),
        ["company_id", "employee_id"],
        "left_anti"
    )
    .count()
)

assert missing_predictions == 0, (
    f"{missing_predictions} ground-truth employees are missing predictions."
)

# --------------------------------------------------
# 4. JOIN GROUND TRUTH WITH INFERRED RELATIONSHIPS
# --------------------------------------------------

evaluated = truth.join(
    predictions,
    ["company_id", "employee_id"],
    "inner"
)

evaluated = evaluated.withColumn(
    "true_relationship_exists",
    F.col("true_manager_id").isNotNull()
).withColumn(
    "predicted_relationship_exists",
    F.col("predicted_manager_id").isNotNull()
).withColumn(
    "correct_relationship",
    F.col("true_manager_id").isNotNull() &
    (F.col("true_manager_id") == F.col("predicted_manager_id"))
).withColumn(
    "false_positive",
    F.col("predicted_manager_id").isNotNull() &
    (
        F.col("true_manager_id").isNull() |
        (F.col("true_manager_id") != F.col("predicted_manager_id"))
    )
).withColumn(
    "false_negative",
    F.col("true_manager_id").isNotNull() &
    (
        F.col("predicted_manager_id").isNull() |
        (F.col("true_manager_id") != F.col("predicted_manager_id"))
    )
)

# --------------------------------------------------
# 5. ADD ORGANIZATIONAL STRUCTURE
# --------------------------------------------------

company_structure = companies.select(
    "company_id",
    "org_structure"
).dropDuplicates()

evaluated = evaluated.join(
    company_structure,
    ["company_id"],
    "left"
)

missing_structure = evaluated.filter(
    F.col("org_structure").isNull()
).count()

assert missing_structure == 0, (
    f"{missing_structure} employees have no organizational structure."
)

# --------------------------------------------------
# 6. METRIC CALCULATION FUNCTION
# --------------------------------------------------

def calculate_metrics(df, group_columns):

    aggregations = [
        F.count("*").alias("total_employees"),

        F.sum(
            F.col("true_relationship_exists").cast("long")
        ).alias("actual_relationships"),

        F.sum(
            F.col("predicted_relationship_exists").cast("long")
        ).alias("predicted_relationships"),

        F.sum(
            F.col("correct_relationship").cast("long")
        ).alias("true_positives"),

        F.sum(
            F.col("false_positive").cast("long")
        ).alias("false_positives"),

        F.sum(
            F.col("false_negative").cast("long")
        ).alias("false_negatives"),

        F.sum(
            F.col("true_relationship_exists").cast("long")
        ).alias("employees_with_actual_managers")
    ]

    if group_columns:
        result = df.groupBy(*group_columns).agg(*aggregations)
    else:
        result = df.agg(*aggregations)

    result = (
        result
        .withColumn(
            "precision",
            F.when(
                F.col("predicted_relationships") > 0,
                F.col("true_positives") /
                F.col("predicted_relationships")
            ).otherwise(F.lit(0.0))
        )
        .withColumn(
            "recall",
            F.when(
                F.col("actual_relationships") > 0,
                F.col("true_positives") /
                F.col("actual_relationships")
            ).otherwise(F.lit(0.0))
        )
        .withColumn(
            "f1_score",
            F.when(
                (F.col("precision") + F.col("recall")) > 0,
                2 * F.col("precision") * F.col("recall") /
                (F.col("precision") + F.col("recall"))
            ).otherwise(F.lit(0.0))
        )
        .withColumn(
            "coverage",
            F.col("predicted_relationships") /
            F.col("total_employees")
        )
    )

    return result

# --------------------------------------------------
# 7. OVERALL EVALUATION
# --------------------------------------------------

overall_metrics = calculate_metrics(evaluated, [])

# --------------------------------------------------
# 8. EVALUATION BY ORGANIZATIONAL STRUCTURE
# --------------------------------------------------

structure_metrics = calculate_metrics(
    evaluated,
    ["org_structure"]
)

# --------------------------------------------------
# 9. SAVE EVALUATION RESULTS
# --------------------------------------------------

(
    overall_metrics
    .withColumn("evaluation_scope", F.lit("OVERALL"))
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(METRICS_TABLE)
)

(
    structure_metrics.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(COMPANY_METRICS_TABLE)
)

# --------------------------------------------------
# 10. DISPLAY RESULTS
# --------------------------------------------------

print("=" * 65)
print("OVERALL MANAGER INFERENCE EVALUATION")
print("=" * 65)

overall_metrics.select(
    "total_employees",
    "actual_relationships",
    "predicted_relationships",
    "true_positives",
    "false_positives",
    "false_negatives",
    "precision",
    "recall",
    "f1_score",
    "coverage"
).show(truncate=False)

print("\nEVALUATION BY ORGANIZATIONAL STRUCTURE")

structure_metrics.select(
    "org_structure",
    "total_employees",
    "actual_relationships",
    "predicted_relationships",
    "true_positives",
    "false_positives",
    "false_negatives",
    "precision",
    "recall",
    "f1_score",
    "coverage"
).orderBy(
    "org_structure"
).show(truncate=False)

print("\nEVALUATION DATA QUALITY")
print("Ground truth duplicate records:", truth_duplicates)
print("Prediction duplicate records:", prediction_duplicates)
print("Missing predictions:", missing_predictions)
print("Missing organizational structure:", missing_structure)

print("\nOUTPUT TABLES")
print(METRICS_TABLE)
print(COMPANY_METRICS_TABLE)

print("\nEVALUATION COMPLETED")

In [0]:
from pyspark.sql import functions as F

GROUND_TRUTH_TABLE = "workspace.firmable_v2_bronze.ground_truth"
HIERARCHY_TABLE = "workspace.firmable_v2_silver.employee_hierarchy"
CANDIDATE_TABLE = "workspace.firmable_v2_silver.manager_candidate_scores"
COMPANY_TABLE = "workspace.firmable_v2_bronze.companies"

# --------------------------------------------------
# 1. LOAD DATA
# --------------------------------------------------

truth = spark.table(GROUND_TRUTH_TABLE).select(
    "company_id",
    "employee_id",
    F.col("actual_manager_id").alias("true_manager_id")
)

hierarchy = spark.table(HIERARCHY_TABLE).select(
    "company_id",
    "employee_id",
    "manager_id",
    "assignment_status",
    "candidate_score",
    "relationship_type",
    "department",
    "inferred_seniority_level",
    "root_type"
)

candidates = spark.table(CANDIDATE_TABLE).select(
    "company_id",
    "employee_id",
    "candidate_manager_id",
    "candidate_score",
    "candidate_rank",
    "relationship_type"
)

companies = spark.table(COMPANY_TABLE).select(
    "company_id",
    "org_structure"
)

# --------------------------------------------------
# 2. REBUILD EMPLOYEE-LEVEL EVALUATION
# --------------------------------------------------

analysis = (
    truth
    .join(hierarchy, ["company_id", "employee_id"], "inner")
    .join(companies, ["company_id"], "left")
    .withColumn(
        "is_false_positive",
        F.col("manager_id").isNotNull() &
        (
            F.col("true_manager_id").isNull() |
            (F.col("manager_id") != F.col("true_manager_id"))
        )
    )
    .withColumn(
        "is_false_negative",
        F.col("true_manager_id").isNotNull() &
        (
            F.col("manager_id").isNull() |
            (F.col("manager_id") != F.col("true_manager_id"))
        )
    )
)

# --------------------------------------------------
# 3. CHECK WHETHER ACTUAL MANAGER WAS A CANDIDATE
# --------------------------------------------------

candidate_match = (
    candidates
    .join(
        truth,
        ["company_id", "employee_id"],
        "inner"
    )
    .groupBy("company_id", "employee_id")
    .agg(
        F.max(
            F.when(
                F.col("candidate_manager_id") == F.col("true_manager_id"),
                1
            ).otherwise(0)
        ).alias("actual_manager_in_candidates"),
        F.max(
            F.when(
                F.col("candidate_rank") == 1,
                F.col("candidate_score")
            )
        ).alias("best_candidate_score"),
        F.max(
            F.when(
                F.col("candidate_rank") == 1,
                F.col("relationship_type")
            )
        ).alias("best_candidate_relationship_type"),
        F.count("*").alias("available_candidate_count")
    )
)

analysis = (
    analysis
    .join(
        candidate_match,
        ["company_id", "employee_id"],
        "left"
    )
    .fillna({
        "actual_manager_in_candidates": 0,
        "available_candidate_count": 0
    })
)

# --------------------------------------------------
# 4. FALSE POSITIVE ANALYSIS
# --------------------------------------------------

print("=" * 70)
print("FALSE POSITIVE ANALYSIS")
print("=" * 70)

fp_by_relationship = (
    analysis
    .filter(F.col("is_false_positive"))
    .groupBy("relationship_type")
    .agg(
        F.count("*").alias("false_positives")
    )
    .orderBy(F.desc("false_positives"))
)

print("\nFALSE POSITIVES BY INFERRED RELATIONSHIP TYPE")
fp_by_relationship.show(30, truncate=False)

fp_by_structure = (
    analysis
    .filter(F.col("is_false_positive"))
    .groupBy("org_structure")
    .count()
    .orderBy(F.desc("count"))
)

print("\nFALSE POSITIVES BY ORGANIZATIONAL STRUCTURE")
fp_by_structure.show(truncate=False)

# --------------------------------------------------
# 5. FALSE NEGATIVE ANALYSIS
# --------------------------------------------------

fn = analysis.filter(F.col("is_false_negative"))

print("\n" + "=" * 70)
print("FALSE NEGATIVE ANALYSIS")
print("=" * 70)

fn_by_status = (
    fn
    .groupBy("assignment_status")
    .agg(
        F.count("*").alias("missed_relationships"),
        F.sum(
            F.col("actual_manager_in_candidates")
        ).alias("actual_manager_was_candidate"),
        F.avg("best_candidate_score").alias("average_best_candidate_score")
    )
    .orderBy(F.desc("missed_relationships"))
)

print("\nFALSE NEGATIVES BY ASSIGNMENT STATUS")
fn_by_status.show(truncate=False)

fn_by_structure = (
    fn
    .groupBy("org_structure")
    .count()
    .orderBy(F.desc("count"))
)

print("\nFALSE NEGATIVES BY ORGANIZATIONAL STRUCTURE")
fn_by_structure.show(truncate=False)

fn_by_department = (
    fn
    .groupBy("department")
    .count()
    .orderBy(F.desc("count"))
)

print("\nFALSE NEGATIVES BY DEPARTMENT")
fn_by_department.show(30, truncate=False)

fn_by_seniority = (
    fn
    .groupBy("inferred_seniority_level")
    .count()
    .orderBy("inferred_seniority_level")
)

print("\nFALSE NEGATIVES BY EMPLOYEE SENIORITY")
fn_by_seniority.show(truncate=False)

# --------------------------------------------------
# 6. CANDIDATE GENERATION RECALL
# --------------------------------------------------

actual_relationships = truth.filter(
    F.col("true_manager_id").isNotNull()
)

candidate_coverage = (
    actual_relationships
    .join(
        candidate_match,
        ["company_id", "employee_id"],
        "left"
    )
    .fillna({
        "actual_manager_in_candidates": 0,
        "available_candidate_count": 0
    })
)

candidate_generation_summary = candidate_coverage.agg(
    F.count("*").alias("actual_relationships"),
    F.sum(
        "actual_manager_in_candidates"
    ).alias("actual_managers_generated_as_candidates"),
    F.sum(
        F.when(
            F.col("actual_manager_in_candidates") == 0,
            1
        ).otherwise(0)
    ).alias("actual_managers_missing_from_candidates")
).withColumn(
    "candidate_generation_recall",
    F.col("actual_managers_generated_as_candidates") /
    F.col("actual_relationships")
)

print("\n" + "=" * 70)
print("CANDIDATE GENERATION RECALL")
print("=" * 70)

candidate_generation_summary.show(truncate=False)

# --------------------------------------------------
# 7. FALSE NEGATIVE ROOT CAUSE CLASSIFICATION
# --------------------------------------------------

fn_root_causes = (
    fn
    .withColumn(
        "root_cause",
        F.when(
            F.col("actual_manager_in_candidates") == 0,
            F.lit("ACTUAL_MANAGER_NOT_GENERATED")
        )
        .when(
            F.col("manager_id").isNotNull(),
            F.lit("WRONG_CANDIDATE_SELECTED")
        )
        .when(
            F.col("assignment_status") == "ABSTAINED_LOW_SCORE",
            F.lit("LOW_CONFIDENCE_THRESHOLD")
        )
        .when(
            F.col("assignment_status") == "UNASSIGNED_CAPACITY_OR_ELIGIBILITY",
            F.lit("MANAGER_CAPACITY_OR_ELIGIBILITY")
        )
        .otherwise(F.lit("OTHER_ASSIGNMENT_GAP"))
    )
)

root_cause_summary = (
    fn_root_causes
    .groupBy("root_cause")
    .count()
    .orderBy(F.desc("count"))
)

print("\n" + "=" * 70)
print("FALSE NEGATIVE ROOT CAUSE BREAKDOWN")
print("=" * 70)

root_cause_summary.show(truncate=False)

# --------------------------------------------------
# 8. SAVE ERROR ANALYSIS TABLES
# --------------------------------------------------

ERROR_ANALYSIS_TABLE = (
    "workspace.firmable_v2_evaluation.error_analysis_employee_level"
)

ROOT_CAUSE_TABLE = (
    "workspace.firmable_v2_evaluation.error_analysis_root_causes"
)

(
    analysis.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(ERROR_ANALYSIS_TABLE)
)

(
    fn_root_causes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(ROOT_CAUSE_TABLE)
)

# --------------------------------------------------
# 9. FINAL SUMMARY
# --------------------------------------------------

print("\n" + "=" * 70)
print("ERROR ANALYSIS COMPLETED")
print("=" * 70)

print("False positives:", analysis.filter(F.col("is_false_positive")).count())
print("False negatives:", fn.count())

print("Employee-level error table:", ERROR_ANALYSIS_TABLE)
print("Root cause table:", ROOT_CAUSE_TABLE)

print("ALL ERROR ANALYSIS TABLES PUBLISHED")